# Environment Setup

Run every cell in this notebook before the first session, and again on any day
whose packages you have not installed yet. It reports which flavour of the Hub
you are running, installs the packages every day needs, checks what each later
day still requires, and confirms that the data for each released day is where
the notebooks expect it.

Read the summary at the bottom. It tells you exactly which command to run for
anything that is missing.

In [ ]:
import importlib
import importlib.metadata
import importlib.util
import platform
import site
import subprocess
import sys
from pathlib import Path

# The Hub offers a CPU flavour and a GPU flavour. They share every base
# package and differ in whether PyTorch is present, so PyTorch identifies them.
FLAVOUR = "GPU" if importlib.util.find_spec("torch") else "CPU"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Hub flavour:", FLAVOUR)

## Installing The Base Packages

`setup/requirements.txt` pins the versions the notebooks are verified against,
and both flavours of the image already provide them, so this normally installs
nothing and simply confirms that the image has not drifted.

Outside a virtual environment the packages go into your home directory, where
they survive the server being restarted. If the install reports that it changed
anything, restart the kernel before running the rest of this notebook, because
installing a different version of a package does not replace one that has
already been imported.

In [ ]:
def install(*arguments):
    """Run pip, installing into the home directory when outside a virtual environment."""
    command = [sys.executable, "-m", "pip", "install"]
    if sys.prefix == sys.base_prefix:
        command.append("--user")
    command += list(arguments)

    print(" ".join(command), "\n")
    completed = subprocess.run(command, capture_output=True, text=True)

    # pip prints one line for every package it did not have to touch, which is
    # every line on an image that already matches, so those are counted rather
    # than shown.
    lines = completed.stdout.strip().splitlines()
    untouched = [line for line in lines if line.startswith("Requirement already satisfied")]
    changed = [line for line in lines if not line.startswith("Requirement already satisfied")]
    if untouched:
        print(f"{len(untouched)} packages were already present at the pinned version.")
    for line in changed:
        print(line)
    if completed.returncode != 0:
        print("\nThe install failed:\n")
        print(completed.stderr.strip())
    return completed.returncode == 0


install("-r", "setup/requirements.txt")

## Packages

The base packages come from `setup/requirements.txt` and every day from 1 to 6
uses them. Each later day adds packages of its own, which you install yourself
from that day's requirements file. PyTorch is the exception: it is large, it is
built against the GPU, and it comes from the image rather than from you.

In [ ]:
BASE_PACKAGES = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]

# Each later day lists the packages you install yourself and the ones that can
# only come from the image.
DAY_PACKAGES = {
    "Day 3": ("setup/requirements-day3.txt", ["cv2"], ["torch", "torchvision"]),
    "Day 4": ("setup/requirements-day4.txt", ["transformers"], ["torch"]),
    "Day 5": ("setup/requirements-day5.txt", ["openai", "langchain", "langgraph"], []),
    "Day 6": ("setup/requirements-day6.txt", ["xgboost"], []),
}

# The name used to import a package is not always the name it is distributed
# under, so the version lookup needs the distribution name.
DISTRIBUTION_NAMES = {"cv2": "opencv-python", "sklearn": "scikit-learn"}


def report(module, note=""):
    """Print one line for a package and say whether it could be imported."""
    if importlib.util.find_spec(module) is None:
        print(f"  MISSING  {module}{note}")
        return False
    try:
        version = importlib.metadata.version(DISTRIBUTION_NAMES.get(module, module))
    except importlib.metadata.PackageNotFoundError:
        version = "unknown"
    print(f"  ok       {module:<14s} {version}")
    return True


missing_base = []
days_to_install = []
needs_gpu_flavour = []

print("Base packages, needed every day")
for module in BASE_PACKAGES:
    if not report(module):
        missing_base.append(module)

for day, (requirements, student_packages, image_packages) in DAY_PACKAGES.items():
    print(f"\n{day}")
    # Every package is reported, so the listing is not cut short at the
    # first one that is absent.
    present = [report(module) for module in student_packages]
    if not all(present):
        days_to_install.append((day, requirements))
    for module in image_packages:
        if not report(module, "  (comes from the GPU flavour)"):
            needs_gpu_flavour.append(day)

## Shadowed Packages

A package installed into your home directory takes precedence over the image's
copy, and your home directory follows you when you switch flavour. That is
useful for the packages you are meant to install, and harmful for the ones you
are not. A PyTorch installed by hand on the CPU flavour will be loaded instead
of the image's CUDA build on the GPU flavour, and the GPU quietly stops being
used.

In [ ]:
try:
    USER_SITE = Path(site.getusersitepackages())
except Exception:
    USER_SITE = None

shadowed = []
if USER_SITE is not None and USER_SITE.is_dir():
    for module in BASE_PACKAGES + ["torch", "torchvision", "scipy", "statsmodels"]:
        spec = importlib.util.find_spec(module)
        if spec is None or not spec.origin:
            continue
        if USER_SITE in Path(spec.origin).parents:
            shadowed.append(module)

if not USER_SITE or not USER_SITE.is_dir():
    print("Nothing is installed in your home directory yet.")
elif shadowed:
    print(f"These are loaded from {USER_SITE} rather than from the image:")
    for module in shadowed:
        print(f"  {module}")
else:
    print(f"No image package is being shadowed by {USER_SITE}.")

## Data Files

Every notebook reads its data through a path relative to itself, so these files
must sit where the check expects them. Days appear in the repository as the
workshop progresses, so a day that has not been released yet is reported as
such rather than as a fault.

In [ ]:
REPOSITORY_ROOT = Path.cwd()

EXPECTED_DATA = {
    "Day 1 - Introduction and Review": ["penguins.csv", "wine_quality.csv"],
    "Day 2 - Data Preparation and Processing": [
        "products.csv",
        "stations.csv",
        "sensors",
    ],
    "Day 3 - Image Processing and Analysis": ["slices.csv", "slices", "masks"],
    "Day 4 - Text Processing and Analysis": [
        "discharge_summaries.csv",
        "abbreviations.csv",
        "icd_codes.csv",
    ],
    "Day 5 - Generative AI and AI Agents": ["sqlite-sakila.db"],
    "Day 6 - Learning With Time-Varying Data": ["data.csv"],
}

absent = []

for day, files in EXPECTED_DATA.items():
    if not (REPOSITORY_ROOT / day).is_dir():
        print(f"pending  {day} has not been released yet")
        continue
    directory = REPOSITORY_ROOT / day / "data"
    if not directory.is_dir():
        print(f"MISSING  {day}/data")
        absent.append(f"{day}/data")
        continue
    for name in files:
        path = directory / name
        if not path.exists():
            print(f"MISSING  {day}/data/{name}")
            absent.append(f"{day}/data/{name}")
            continue
        if path.is_dir():
            size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
        else:
            size = path.stat().st_size
        readable = f"{size / 1e6:.1f} MB" if size >= 1e6 else f"{size / 1e3:.0f} kB"
        print(f"ok       {day}/data/{name}  ({readable})")

## Accelerator

Days 3 and 4 train neural networks with PyTorch. They will run without a GPU,
but a training step that takes a minute on a GPU can take most of an hour on a
processor alone.

In [ ]:
if FLAVOUR == "CPU":
    print("This is the CPU flavour, so there is no GPU. Days 3 and 4 want the GPU flavour.")
else:
    import torch

    if torch.cuda.is_available():
        print(f"PyTorch {torch.__version__} sees a GPU: {torch.cuda.get_device_name(0)}")
    else:
        print(f"PyTorch {torch.__version__} is installed but sees no GPU.")

## Summary

In [ ]:
USER_FLAG = "" if sys.prefix != sys.base_prefix else "--user "

if not (missing_base or days_to_install or needs_gpu_flavour or shadowed or absent):
    print("Everything the released material needs is present. You are ready to start.")

if missing_base:
    print("Missing base packages, which the install cell above should have provided:")
    for module in missing_base:
        print(f"  {module}")
    print("Rerun that cell, restart the kernel, and run this notebook again.\n")

if days_to_install:
    print("Days whose packages you still need to install:")
    for day, requirements in days_to_install:
        print(f"  {day}:  pip install {USER_FLAG}-c setup/requirements.txt -r {requirements}")
    print()

if needs_gpu_flavour:
    days = sorted(set(needs_gpu_flavour))
    # One day takes a singular verb, and three or more need a conjunction
    # before the last of them so the sentence does not trail off in commas.
    if len(days) == 1:
        listed = days[0]
    elif len(days) == 2:
        listed = f"{days[0]} and {days[1]}"
    else:
        listed = ", ".join(days[:-1]) + f", and {days[-1]}"
    verb = "needs" if len(days) == 1 else "need"
    print(f"{listed} {verb} PyTorch, which only the GPU flavour provides.")
    print("Stop your server from the Hub control panel and start the GPU flavour.")
    print("Do not install PyTorch yourself.\n")

if shadowed:
    print("These packages are loaded from your home directory instead of the image:")
    for module in shadowed:
        print(f"  {module}")
    print("That overrides the image on both flavours. Unless you meant to do it,")
    print(f"remove them with:  pip uninstall {' '.join(shadowed)}\n")

if absent:
    print("Missing data:")
    for item in absent:
        print(f"  {item}")
    print("Show this to the assistant before the session that needs it.")